# Libraries

In [2]:
import pandas as pd
import networkx as nx
import numpy as np
import osmnx as ox
import geopandas as gpd
import pickle
from tqdm import tqdm
import os

In [3]:
# Set home directory
home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Global road network resilience/01_data' # CURA
home_dir = '/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology(2)/Research/Global road network resilience/01_data' # SCARP

# Validation

## Road segment height

In this project, we extract the 

### Step 1: Sample 5 networks for each World Bank region (7 regions, 35 networks worldwide)

In [ ]:
# Read settlement cluster shapefiles
clusters_2564_dir = "/final_bnd/poly_2564_WB_regions.shp"
clusters_2564 = gpd.read_file(home_dir + clusters_2564_dir)

# Number of random network IDs to select per World Bank region
n_samples = 5

# Group by 'REGION_WB' and sample 'n_samples' random rows from each group
# The result will be a new DataFrame containing the sampled rows.
random_network_ids_per_region = clusters_2564.groupby('REGION_WB').sample(n=n_samples, random_state=2025)

# Save sampled network ids in csv file
random_network_ids_per_region.to_csv(home_dir+"/road_elevation_validation/sampled_35networks.csv")

In [66]:
# Print only the net_id and region
print("\nSampled 'net_id' for each region:")
print(random_network_ids_per_region[['REGION_WB', 'net_id']])


Sampled 'net_id' for each region:
                       REGION_WB  net_id
599          East Asia & Pacific  1679.0
1338         East Asia & Pacific  2849.0
1545         East Asia & Pacific  3225.0
2160         East Asia & Pacific  4215.0
1444         East Asia & Pacific  3021.0
2598       Europe & Central Asia   847.0
121        Europe & Central Asia  1143.0
1668       Europe & Central Asia   343.0
1805       Europe & Central Asia   368.0
1346       Europe & Central Asia   287.0
1639   Latin America & Caribbean  3366.0
894    Latin America & Caribbean  2002.0
1615   Latin America & Caribbean  3328.0
1965   Latin America & Caribbean  3962.0
2070   Latin America & Caribbean  4103.0
929   Middle East & North Africa  2062.0
452   Middle East & North Africa  1521.0
833   Middle East & North Africa  1926.0
1161  Middle East & North Africa  2550.0
932   Middle East & North Africa  2066.0
686                North America  1753.0
394                North America  1456.0
174                Nor

### Step 2: Extract OSM road networks

In [ ]:
# CSV file with sampled network IDs
random_network_ids_per_region = pd.read_csv(home_dir+"/road_elevation_validation/sampled_35networks.csv", index_col=0)

# Define coordinate reference system: WGS84 - World Geodetic System 1984 used in GPS
crs_lonlat = {'init': 'epsg:' + str(4326)}

# Read in the shapefile with the convex hulls of the settlement cluster boundaries
convex_hull_2564_shp = gpd.read_file(home_dir + '/final_bnd/poly_2564_WB_regions.shp').to_crs(crs_lonlat)

# Loop through the sampled network ids and extract OSM road networks
for net_id in random_network_ids_per_region.net_id.values:
    # Select convexhull poly
    convex_hull_sel = convex_hull_2564_shp[convex_hull_2564_shp["net_id"] == net_id].iloc[0]
    G = ox.graph_from_polygon(convex_hull_sel.geometry, network_type='drive_service')
    with open(home_dir + '/road_elevation_validation/sampled_networks/g_raw_drive_conv_' + str(net_id) + '.pk', 'wb') as handle:
        pickle.dump(G, handle, protocol=2)

OPTIONAL: examine road types in raw OSM networks

In [197]:
# [OPTIONAL] This cell examines the highway classes or road types
# Initiate dataframe to store road types
net_type_df = pd.DataFrame(columns=['net_id', 'net_types'], index=range(35))
counter = 0
net_type_lst = [] # Initiate list of network types

# Examine the road types
# Function to remove list nestings in highway road classes
def removeNestings(l): 
    output = []
    for i in l: 
        if type(i) == list: 
            output.extend(removeNestings(i))  # Use extend to add flattened results
        else: 
            output.append(i)
    return output

for file in os.listdir(home_dir + '/road_elevation_validation/sampled_networks/'):
    if file.endswith('.pk'):
        G = pickle.load(open(home_dir + '/road_elevation_validation/sampled_networks/' + file, 'rb'))
        net_id = int(file.split("_")[-1].split(".")[0])
        net_type_df.at[counter, 'net_id'] = net_id
        # Highway class unique values
        highway_class_dict = nx.get_edge_attributes(G,'highway')
        highway_class_lst = []
        for item in highway_class_dict.values():
            highway_class_lst.append(item)
        net_type_df.at[counter, 'net_types'] = list(set(removeNestings(highway_class_lst)))
        counter += 1
        net_type_lst.append(list(set(removeNestings(highway_class_lst))))

Below a dataframe of network id and unique road network types is created.

In [201]:
net_type_df.head(5)

,net_id,net_types
0,1521,"[tertiary, living_street, tertiary_link, resid..."
1,2550,"[tertiary, tertiary_link, residential, trunk_l..."
2,287,"[tertiary, living_street, tertiary_link, resid..."
3,3910,"[tertiary, living_street, residential, trunk_l..."
4,3481,"[tertiary, living_street, tertiary_link, resid..."


These are the unique road network types for all 35 sampled networks

In [207]:
set(removeNestings(net_type_lst))

{'yes', 'trunk_link', 'secondary_link', 'escape', 'step', 'tertiary_link', 'trunk', 'bus_stop', 'residential', 'motorway_link', 'primary_link', 'road', 'tertiary', 'living_street', 'service', 'crossing', 'unclassified', 'motorway', 'secondary', 'primary', 'busway'}

Remove highway classes that cause data bias

In [229]:
# Identify highway classes to remove
highway_classes_to_remove = [
    'yes',
    'escape',
    'step',
    'tertiary_link',
    'bus_stop',
    'residential',
    'tertiary',
    'living_street',
    'service',
    'crossing',
    'unclassified',
    'busway'
                             ]
for file in os.listdir(home_dir + '/road_elevation_validation/sampled_networks/'):
    if file.endswith('.pk'):
        net_id = int(file.split('_')[-1].split('.')[0])
        G = pickle.load(open(home_dir + '/road_elevation_validation/sampled_networks/' + file, 'rb'))
        G_lite = G.copy()
        for i,j,data in G.edges.data():
            highway_cls = data['highway']
            if highway_cls in highway_classes_to_remove:
                G_lite.remove_edge(i, j)
        # Remove isolated nodes as a result of the edge removal
        G_lite.remove_nodes_from(list(nx.isolates(G_lite)))
        with open(home_dir + '/road_elevation_validation/sampled_networks_lite/graph_cov_lite_' + str(int(net_id)) + '.pk', 'wb') as handle:
                pickle.dump(G_lite, handle, protocol=2)


### Step 3: Attach elevation values to road network

In [ ]:
# Extract road elevation using Google API
for file in os.listdir(home_dir + '/road_elevation_validation/sampled_networks_lite/'):
    if file.endswith('.pk'):
        net_id = int(file.split('_')[-1].split('.')[0])
        G = pickle.load(open(home_dir + '/road_elevation_validation/sampled_networks_lite/' + file, 'rb'))
        # add elevation to each of the nodes, using the google elevation API, then calculate edge grades
        G = ox.elevation.add_node_elevations_google(G, api_key='your API key here')
        G = ox.add_edge_grades(G)
        # Save as graphml
        ox.io.save_graphml(G, home_dir + f'/road_elevation_validation/sampled_network_lite_elev_graphml/graph_cov_lite_elev_{net_id}.graphml')
        # Save as geopackage
        ox.io.save_graph_geopackage(G, home_dir + f"/road_elevation_validation/sampled_network_lite_elev_gpkg/graph_cov_lite_elev_{net_id}.gpkg")

## Road closures (Houston, USA)

In [3]:
houston_graph_10flood = pickle.load(open(home_dir + '/Harvey_validation/G_cov_no_r_2139_10flooded.pk', 'rb'))

/var/folders/nc/ywj4rgkn46l95_tqft45f5640000gs/T/ipykernel_19125/4091128666.py:1: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again as this compatibility may be removed in a future version of shapely.
  houston_graph_10flood = pickle.load(open(home_dir + '/Harvey_validation/G_cov_no_r_2139_10flooded.pk', 'rb'))


In [ ]:
# Critical inundation thresholds
gamma_lst = [0.3, 0.25, 0.2, 0.15]

# Flood return period
rp = 1000

for gamma in gamma_lst:
    # Make a copy of the graph with flood inundation values
    houston_disrupted_graph = houston_graph_10flood.copy()
    # Delete nodes with flood inundation greater than 30mm or 0.3m
    for node_id, node_data in houston_graph_10flood.nodes.data():
        if node_data['FUP_' + str(rp)] >= gamma:
            houston_disrupted_graph.remove_node(node_id)
    # Delete edges with flood inundation greater than 30mm or 0.3m
    houston_disrupted_graph2 = houston_disrupted_graph.copy()
    for start_id, end_id, edge_data in houston_disrupted_graph.edges.data():
        try:
            if edge_data['FUP_' + str(rp)] >= gamma:
                houston_disrupted_graph2.remove_edge(start_id, end_id)
        except KeyError:
            continue

## Travel disruption (Nanjing, China)

### Build network model

In [8]:
# Define coordinate reference system: WGS84 - World Geodetic System 1984 used in GPS
crs_lonlat = {'init': 'epsg:' + str(4326)}

# Read in the shapefile with the convex hulls of the settlement cluster boundaries
nanjing_bnd_shp = gpd.read_file(home_dir + '/01_public/Nanjing_validation/Nanjing_city_bnd.shp').to_crs(crs_lonlat)

# Extract OSM road network for Nanjing City
poly = nanjing_bnd_shp.geometry.iloc[0]
G = ox.graph_from_polygon(poly, network_type='drive_service')
with open(home_dir + '/01_public/Nanjing_validation/nj_raw_drive.pk', 'wb') as handle:
    pickle.dump(G, handle, protocol=2)

/opt/miniconda3/envs/global0/lib/python3.12/site-packages/pyproj/crs/crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


In [ ]:
# Function to remove list nestings in highway road classes
def removeNestings(l): 
    output = []
    for i in l: 
        if type(i) == list: 
            output.extend(removeNestings(i))  # Use extend to add flattened results
        else: 
            output.append(i)
    return output

# Highway class unique values
highway_class_dict = nx.get_edge_attributes(G,'highway')
highway_class_lst = []
for item in highway_class_dict.values():
    highway_class_lst.append(item)

# Identify highway classes to remove
highway_classes_to_remove = [
    'tertiary_link',
    'road',
    'residential',
    'tertiary',
    'living_street',
    'service',
    'unclassified'
                             ]

# Delete 'highway' classes that are not included in the scope of study
G_lite = G.copy()
for i,j,data in G.edges.data():
    highway_cls = data['highway']
    if highway_cls in highway_classes_to_remove:
        G_lite.remove_edge(i, j)
    # Remove isolated nodes as a result of the edge removal
    G_lite.remove_nodes_from(list(nx.isolates(G_lite)))
    # Save resulting graph as pickle file
    with open(home_dir + '/01_public/Nanjing_validation/nj_lite.pk', 'wb') as handle:
        pickle.dump(G_lite, handle, protocol=2)

# Save as geopackage
ox.io.save_graph_geopackage(G_lite, home_dir + '/01_public/Nanjing_validation/nj_lite.gpkg')

# add elevation to each of the nodes, using the google elevation API, then calculate edge grades
G_lite = ox.elevation.add_node_elevations_google(G_lite, api_key='your api key here')
G_lite = ox.add_edge_grades(G_lite)

# Save as graphml
ox.io.save_graphml(G_lite, home_dir + '/01_public/Nanjing_validation/nj_lite_elev.graphml')
# Save as geopackage
ox.io.save_graph_geopackage(G_lite, home_dir + "/01_public/Nanjing_validation/nj_lite_elev.gpkg")

### Fathom flood inundation

In [ ]:
# Read graph from graphml file
G_lite_elev = ox.io.load_graphml(filepath=home_dir + "/01_public/Nanjing_validation/nj_lite_elev.graphml")

# Spatial overlay with Fathom flood maps
# Flood the nodes with 10 flood return periods
RP_lst = [5, 10, 20, 50, 75, 100, 200, 250, 500, 1000]
for file in tqdm(os.listdir('L:/yiyi/grth_90_graphs_drive_service_no_res/')):
    net_id = int(file[15:][:-3])
    G = pickle.load(open('L:/yiyi/grth_90_graphs_drive_service_no_res/'+ file, 'rb'))
    G_flooded = G.copy()
    # Flood the nodes with 10 scenarios
    for rp in RP_lst:
        G_flooded = gn.sample_raster(G_flooded, 'J:/yiyi/mosaics/FUP'+ str(rp) +'_mosaic.tif', 'FUP_'+ str(rp))
    # Now we have flooded nodes with 10 flood depths
    # Flood the edges too by assigning the larger nodes flood depth to edge
    for i, j, data in G_flooded.edges.data():
        for rp in RP_lst:
            FUP_i = G_flooded.nodes[i]['FUP_'+str(rp)]
            FUP_j = G_flooded.nodes[j]['FUP_'+str(rp)]
            G_flooded[i][j][0]['FUP_'+str(rp)] = max(FUP_i, FUP_j)
